# Chapter 7: Competitive Intelligence and Market Access

This notebook executes the chapter as one decision chain. It uses fictional products, patients, payers, accounts, and planted synthetic events.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "ch07_competitive").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ch07_competitive.scripts.run_analysis import run_analysis  # noqa: E402

results = run_analysis(ROOT)
print("Loaded Chapter 7 evidence package.")


Loaded Chapter 7 evidence package.


## 7.1 Opening evidence

The corrected treatment cohort stays aligned with Chapter 5.


In [2]:
headline = results["headline"].iloc[0]
print(f"New-to-therapy patients: {int(headline.new_to_therapy_patients):,}")
print(f"Roventra new starts: {int(headline.roventra_new_starts):,}")
print(
    f"Materially restricted lives: {int(headline.restricted_lives):,} "
    f"of {int(headline.total_lives):,} "
    f"({headline.restricted_lives / headline.total_lives:.1%})"
)
print(f"Payer-region access flags: {int(headline.payer_region_access_flags)} of 32")
print(f"Payer-region adoption flags: {int(headline.payer_region_adoption_flags)} of 32")


New-to-therapy patients: 3,415
Roventra new starts: 2,798
Materially restricted lives: 6,740,000 of 10,926,000 (61.7%)
Payer-region access flags: 20 of 32
Payer-region adoption flags: 3 of 32


The 3,415-patient cohort and 2,798 Roventra starts match the Chapter 5 washout-corrected line table.


## 7.2 Effective-dated access and covered lives


In [3]:
summary = results["covered_lives_summary"].query("payer_type == 'All'").copy()
summary["plan_coverage_rate"] = summary.plan_coverage_rate.map(lambda v: f"{v:.1%}")
summary["covered_lives_rate"] = summary.covered_lives_rate.map(lambda v: f"{v:.1%}")
summary["unrestricted_lives_rate"] = summary.unrestricted_lives_rate.map(
    lambda v: f"{v:.1%}"
)
summary["access_quality_score"] = summary.access_quality_score.map(lambda v: f"{v:.3f}")
print(summary.to_string(index=False))
print()
restriction_lives = results["restriction_lives"].copy()
restriction_lives["lives_share"] = restriction_lives.lives_share.map(
    lambda v: f"{v:.1%}"
)
print(restriction_lives.to_string(index=False))


payer_type  plans  covered_plans plan_coverage_rate  total_lives  covered_lives covered_lives_rate  unrestricted_lives unrestricted_lives_rate access_quality_score
       All     32             24              75.0%     10926000        8314000              76.1%                   0                    0.0%                0.516

       access_state  payer_region_cells  enrolled_lives lives_share
Prior authorization                  12         4186000       38.3%
          Step edit                  12         4128000       37.8%
        Non-covered                   8         2612000       23.9%


Plan coverage and covered lives use different denominators. Every covered cell retains at least 1 utilization-management restriction in this synthetic case.


## 7.3 Corrected competitive starts


In [4]:
print(results["source_of_business"].to_string(index=False))
print()
line1 = results["corrected_line1"]
mix = (
    line1.groupby("first_regimen").patient_id.nunique()
    .sort_values(ascending=False)
    .rename("patients")
    .reset_index()
)
mix["share"] = (mix.patients / len(line1)).map(lambda v: f"{v:.1%}")
print(mix.to_string(index=False))


            source_of_business  patients  brand_new_starts
                New to therapy      3415              2798
Continuing after washout check       513                 0
                        Switch        24                 0
                        Add on         4                 0
                       Restart         0                 0

   first_regimen  patients share
        Roventra      2798 81.9%
          Vexpro       309  9.0%
         Nexoral       303  8.9%
Nexoral + Vexpro         5  0.1%


The source-of-business table keeps 24 switches and 4 additions in separate categories.


## 7.4 Prescription attempts


In [5]:
trace = results["pat02034_attempt_trace"]
columns = [
    "fill_number", "first_submission_date", "last_transaction_date",
    "transaction_rows", "had_pend", "had_reversal",
    "final_outcome", "days_to_paid",
]
print(trace[columns].to_string(index=False))


 fill_number first_submission_date last_transaction_date  transaction_rows  had_pend  had_reversal final_outcome  days_to_paid
           0            2024-07-02            2024-07-09                 2      True         False     Completed           7.0
           1            2024-08-09            2024-08-15                 3     False          True     Completed           0.0
           2            2024-09-09            2024-09-09                 1     False         False     Completed           0.0
           3            2024-10-09            2024-10-09                 1     False         False     Completed           0.0


Seven transaction rows collapse into 4 completed attempts for PAT02034.


## 7.5 Access and adoption decisions


In [6]:
decisions = results["payer_region_decisions"]
selected = (
    decisions.set_index(["payer_id", "region"])
    .loc[[("PAY002", "Northeast"), ("PAY004", "Midwest"), ("PAY005", "South")]]
    .reset_index()
)
selected["brand_share"] = selected.brand_share.map(lambda v: f"{v:.1%}")
selected["probability_below_benchmark"] = selected.probability_below_benchmark.map(
    lambda v: f"{v:.1%}"
)
print(selected[[
    "payer_id", "region", "access_state", "treated_patients",
    "brand_starts", "brand_share", "probability_below_benchmark", "action",
]].to_string(index=False))
print()
print(decisions.action.value_counts().to_string())


payer_id    region        access_state  treated_patients  brand_starts brand_share probability_below_benchmark          action
  PAY002 Northeast         Non-covered               100            85       85.0%                       23.7%     Access work
  PAY004   Midwest Prior authorization               118            91       77.1%                       87.1% Adoption review
  PAY005     South         Non-covered               129            97       75.2%                       95.2% Dual workstream

action
Access work         19
Defend and learn    10
Adoption review      2
Dual workstream      1


The 3 selected cells demonstrate access work, adoption review, and a dual workstream.


## 7.6 Controlled formulary-event measurement


In [7]:
event = results["formulary_event_effect"].iloc[0]
print(f"Immediate level effect: {event.immediate_effect:+.1%}")
print(f"Slope change per week: {event.slope_change_per_week:+.2%}")
print(
    f"Week {int(event.effect_week)} effect: {event.effect_at_week:+.1%} "
    f"(95% CI {event.effect_at_week_lower_95:+.1%} "
    f"to {event.effect_at_week_upper_95:+.1%})"
)
diagnostic = results["synthetic_control_diagnostics"].iloc[0]
print(f"Pre-period RMSPE: {diagnostic.pre_rmspe:.3f}")
print(f"Post-period mean gap: {diagnostic.post_mean_gap:+.1%}")


Immediate level effect: +7.4%
Slope change per week: +0.24%
Week 28 effect: +10.0% (95% CI +6.1% to +14.0%)
Pre-period RMSPE: 0.038
Post-period mean gap: +7.5%


The controlled ITS recovers the planted level and slope improvement. The synthetic control uses independent donor series.


## 7.7 Account actions


In [8]:
accounts = results["account_access_adoption_actions"]
print(accounts.action.value_counts().to_string())
print()
print(
    accounts.set_index("account_id")
    .loc[["ACC155", "ACC005", "ACC121"], [
        "attributed_patients", "treated_patients", "brand_starts",
        "restricted_patients", "action", "reason_code",
    ]]
    .to_string()
)


action
Monitor             65
Access work         24
Defend and learn    22
Dual workstream      2
Adoption review      1

            attributed_patients  treated_patients  brand_starts  restricted_patients            action                        reason_code
account_id                                                                                                                               
ACC155                     38.0              24.0          16.0                 22.0   Adoption review               ACCOUNT_ADOPTION_GAP
ACC005                     24.0              11.0           6.0                 18.0   Dual workstream   DUAL_ACCOUNT_ACCESS_AND_ADOPTION
ACC121                     34.0              15.0          12.0                 20.0  Defend and learn  ACCOUNT_DEFEND_SUPPORTED_ADOPTION


The account table reuses the Chapter 6 patient-HCP-account mapping and preserves mixed payer evidence.


## 7.8 Monitoring and evidence sufficiency


In [9]:
alerts = results["changepoint_alerts"].head(3).copy()
alerts["standardized_cusum"] = alerts.standardized_cusum.round(3)
print(alerts.to_string(index=False))
print()
print(results["switch_evidence"][[
    "first_regimen", "patients", "switch_events",
    "median_time_to_switch", "comparison_status",
]].to_string(index=False))


 week direction  standardized_cusum
   20  Increase               4.136
   24  Increase               4.127
   32  Increase               4.538

   first_regimen  patients  switch_events median_time_to_switch          comparison_status
        Roventra      2798              0           Not reached Insufficient switch events
          Vexpro       309             12           Not reached Insufficient switch events
         Nexoral       303             12           Not reached Insufficient switch events
Nexoral + Vexpro         5              0           Not reached Insufficient switch events


CUSUM supplies a review date. The switch table records that comparative medians are not reached.
